In [9]:
import os
from typing import TypedDict, Dict
from dotenv import load_dotenv
from langchain_cohere import ChatCohere
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END

# Load environment variables
load_dotenv('config.env')
assert os.getenv('COHERE_API_KEY') is not None
assert os.getenv('TAVILY_API_KEY') is not None

# Initialize Cohere LLM and Tavily Search Tool
# 'command-r-08-2024' or 'command-r' is highly recommended for structured/factual tasks
llm = ChatCohere(model="command-r-plus-08-2024", temperature=0)
search_tool = TavilySearchResults(max_results=3)

In [2]:
def collect_topic(state: Dict) -> Dict:
    # Requirement: Ask the patient what health topic they'd like to learn about
    print("\n--- Welcome to MediTech HealthBot ---")
    topic = input("What health topic or medical condition would you like to learn about today? \n> ")
    return {"topic": topic}

def search_medical_info(state: Dict) -> Dict:
    # Requirement: Search Tavily focusing on reputable medical sources
    topic = state["topic"]
    print(f"\n[Searching for reputable medical information on '{topic}'...]")
    # Appending context to guide Tavily towards medical sources
    query = f"{topic} medical condition symptoms treatment causes reputable sources"
    search_results = search_tool.invoke({"query": query})
    return {"search_results": str(search_results)}

def generate_summary(state: Dict) -> Dict:
    # Requirement: Summarize search results in 3-4 patient-friendly paragraphs using NO other knowledge
    prompt = f"""
    You are a medical assistant explaining concepts to a patient. 
    Using ONLY the following search results, provide a 3-4 paragraph patient-friendly explanation of the topic. 
    Do not use any external knowledge. If the context does not contain enough information, state that.
    
    Search Results: {state['search_results']}
    """
    response = llm.invoke(prompt)
    summary = response.content
    print("\n--- Summary ---")
    print(summary)
    print("---------------")
    return {"summary": summary}

def prompt_readiness(state: Dict) -> Dict:
    # Requirement: Prompt patient to indicate when they're ready for a comprehension check
    ready = ""
    while ready.lower() not in ['y', 'yes']:
        ready = input("\nAre you ready for a quick comprehension check? (y/yes) \n> ")
    return {"is_ready_for_quiz": True}

def create_quiz(state: Dict) -> Dict:
    # Requirement: Generate a single relevant quiz question based on the provided information alone
    prompt = f"""
    Based ONLY on the summary provided below, create a single, clear quiz question to test the patient's comprehension. 
    Do not include the answer in your response, just the question.
    
    Summary: {state['summary']}
    """
    response = llm.invoke(prompt)
    return {"quiz_question": response.content}

def administer_quiz(state: Dict) -> Dict:
    # Requirement: Present question and collect answer
    print("\n--- Comprehension Check ---")
    print(state["quiz_question"])
    user_answer = input("\nYour answer: \n> ")
    return {"user_answer": user_answer}

def grade_quiz(state: Dict) -> Dict:
    # Requirement: Evaluate response, provide a grade (A, B, C, etc.), and explanation with citations from the summary
    prompt = f"""
    You are a medical educator. Grade the patient's answer to the quiz question based ONLY on the summary provided.
    
    Summary: {state['summary']}
    Quiz Question: {state['quiz_question']}
    Patient's Answer: {state['user_answer']}
    
    Provide a letter grade (e.g., A, B, C, D, F) and a justification for the grade. 
    Your justification MUST include direct quotes/citations from the Summary to reinforce learning.
    """
    response = llm.invoke(prompt)
    feedback = response.content
    print("\n--- Quiz Results ---")
    print(feedback)
    print("--------------------")
    return {"grade_feedback": feedback}

def check_continuation(state: Dict) -> Dict:
    # Requirement: Ask if patient wants a new topic or to exit. Reset state if new topic to maintain privacy.
    choice = input("\nWould you like to learn about another health topic? (y/n) \n> ")
    if choice.lower() in ['y', 'yes']:
        # Return a completely cleared state dict to reset for the next run
        return {
            "topic": "",
            "user_answer": "",
            "search_results": "",
            "summary": "",
            "quiz_question": "",
            "grade_feedback": "",
            "is_ready_for_quiz": False,
            "continue_session": True
        }
    else:
        print("\nThank you for using MediTech HealthBot. Goodbye!")
        return {"continue_session": False}

In [5]:
from typing import TypedDict

class HealthBotState(TypedDict):
    # Patient inputs
    topic: str                 # The health topic entered by the patient
    user_answer: str           # The patient's answer to the quiz
    
    # LLM & Tool outputs
    search_results: str        # Raw context retrieved from Tavily
    summary: str               # The 3-4 paragraph patient-friendly summary
    quiz_question: str         # The single generated quiz question
    grade_feedback: str        # The final grade and citation-backed justification
    
    # Workflow control
    is_ready_for_quiz: bool    # Tracks if the patient indicated they are ready
    continue_session: bool     # Tracks if the patient wants to learn a new topic

In [10]:
# Conditional edge routing function
def route_continuation(state: Dict):
    if state.get("continue_session"):
        return "collect_topic"
    return END

# Build the graph
workflow = StateGraph(HealthBotState)

# Add all nodes
workflow.add_node("collect_topic", collect_topic)
workflow.add_node("search_medical_info", search_medical_info)
workflow.add_node("generate_summary", generate_summary)
workflow.add_node("prompt_readiness", prompt_readiness)
workflow.add_node("create_quiz", create_quiz)
workflow.add_node("administer_quiz", administer_quiz)
workflow.add_node("grade_quiz", grade_quiz)
workflow.add_node("check_continuation", check_continuation)

# Define sequential edges
workflow.add_edge("collect_topic", "search_medical_info")
workflow.add_edge("search_medical_info", "generate_summary")
workflow.add_edge("generate_summary", "prompt_readiness")
workflow.add_edge("prompt_readiness", "create_quiz")
workflow.add_edge("create_quiz", "administer_quiz")
workflow.add_edge("administer_quiz", "grade_quiz")
workflow.add_edge("grade_quiz", "check_continuation")

# Define conditional edge for resetting or ending
workflow.add_conditional_edges(
    "check_continuation",
    route_continuation
)

# Set the entry point and compile
workflow.set_entry_point("collect_topic")
healthbot_app = workflow.compile()

# Execution trigger (to run in your Jupyter Notebook)
if __name__ == "__main__":
    # Initialize with an empty state
    initial_state = {
        "topic": "",
        "user_answer": "",
        "search_results": "",
        "summary": "",
        "quiz_question": "",
        "grade_feedback": "",
        "is_ready_for_quiz": False,
        "continue_session": False
    }
    # Stream the graph execution
    healthbot_app.invoke(initial_state, config={"recursion_limit": 50})


--- Welcome to MediTech HealthBot ---

[Searching for reputable medical information on 'diabetis'...]

--- Summary ---
Diabetes is a serious medical condition that affects millions of people worldwide. It occurs when your blood sugar, or glucose, is consistently too high. This happens when your body doesn't produce enough insulin, a hormone that helps regulate blood sugar, or when your body doesn't use insulin effectively. Diabetes is a chronic condition, meaning it is long-lasting and often requires lifelong management.

There are several types of diabetes, but the most common form is Type 2 diabetes, which accounts for 90-95% of all diabetes cases. In Type 2 diabetes, your body becomes resistant to insulin, or doesn't produce enough insulin to maintain normal blood sugar levels. This can lead to a range of symptoms, such as increased thirst, frequent urination, and slow-healing cuts and sores. If left untreated or poorly managed, diabetes can cause various complications. These inclu